# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

To navigate the dataset, first list all record sets (by `@id`). Then, list the fields and columns for each record set for further exploration.

In [ ]:
# List all record sets by their @id and display basic info
print("Available record sets:")
record_sets = []
for rs in dataset.metadata.record_sets:
    print(f"- @id: {rs.id}, name: {rs.name if hasattr(rs, 'name') else 'N/A'}")
    record_sets.append(rs.id)
    # List fields and their @id within this record set
    print("  Fields and columns:")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"    Field @id: {field.id}, Name: {getattr(field, 'name', 'N/A')}, Data type: {getattr(field, 'data_type', 'N/A')}")
            if hasattr(field, 'column') and field.column is not None:
                print(f"      Column(s) @id: {field.column}")
            else:
                print(f"      Column(s) @id: N/A")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

You can choose a record set to explore in detail. In this example, we choose the first record set found in the metadata (change as appropriate if your target is different).

In [ ]:
# Extract data from all available record sets and store in a dict of DataFrames, using @id as keys
dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set @id: {record_set_id}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# Example: See the available columns for the first loaded record set
if record_sets:
    example_record_set = record_sets[0]
    print(f"Columns for record set {example_record_set}: ", dataframes[example_record_set].columns.tolist())
    display(dataframes[example_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In this example, we programmatically select the first numeric field for analyses.

In [ ]:
import numpy as np

# Identify a numeric field from first record set
numeric_field = None
rs_metadata = None
for rs in dataset.metadata.record_sets:
    if rs.id == example_record_set:
        rs_metadata = rs
        break
if rs_metadata is not None and hasattr(rs_metadata, 'fields'):
    for field in rs_metadata.fields:
        # Typical numeric types in croissant: 'Float', 'Integer', 'Number'
        typ = getattr(field, 'data_type', None)
        if typ is not None and (('Float' in typ) or ('Integer' in typ) or ('Number' in typ)):
            numeric_field = field.id
            break

if numeric_field is None:
    raise ValueError("No numeric field found in the selected record set.")

print(f"Using numeric field for EDA: {numeric_field}")

# Filter, normalize, and group data
df = dataframes[example_record_set]
if numeric_field not in df.columns:
    raise ValueError(f"Field {numeric_field} not found in DataFrame columns: {df.columns}")
# Convert to numeric (if string)
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
threshold = df[numeric_field].mean()  # As example, use mean as threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
display(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to find a groupable field (categorical)
group_field = None
if rs_metadata is not None:
    for field in rs_metadata.fields:
        typ = getattr(field, 'data_type', None) or ''
        nm = getattr(field, 'id', None)
        if nm != numeric_field and ('Text' in typ or 'String' in typ or 'Category' in str(typ)):
            group_field = nm
            break

if group_field is not None and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot the distribution of the selected numeric field and the normalized values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot of the numeric field
plt.figure(figsize=(10,5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of field: {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# If group field exists, visualize group means
if group_field is not None and group_field in df.columns:
    group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
    plt.figure(figsize=(14,5))
    sns.barplot(x=group_means.index.astype(str), y=group_means.values)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset
"Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells"
using the `mlcroissant` library. We reviewed the available record sets, extracted a DataFrame, performed basic filtering and normalization on a numeric field, and visualized the data distribution. For further analysis, see the full Croissant schema metadata and explore additional record sets and relationships.